# 02 — COLMAP

COLMAPは、多視点画像から各画像のカメラの外部・内部パラメータと3D点群を求めます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.io as pio

from gs_tutorial.colmap import (
    ColmapPaths,
    colmap_database_summary,
    colmap_stage_status,
    export_processed_text,
    export_sparse_text,
    extract_features,
    map_sparse,
    match_features,
    read_feature_keypoints,
    read_strongest_verified_pair,
    resolve_backend,
    select_sparse_model,
    sparse_model_summary,
    undistort_images,
)
from gs_tutorial.config import load_config
from gs_tutorial.dataset import load_colmap_text, scene_summary
from gs_tutorial.visualization import (
    feature_keypoints_figure,
    sparse_scene_figure,
    verified_matches_figure,
)

pio.renderers.default = 'notebook'

cwd = Path.cwd().resolve()
repo = cwd.parent if (cwd.parent / "pyproject.toml").is_file() else cwd
config_path = repo / "configs/video.yaml"  # Use images.yaml for multi-view image input
cfg = load_config(config_path)
paths = ColmapPaths.create(repo / cfg.project_dir / "colmap", repo / cfg.project_dir / "images")
backend = resolve_backend("auto")
print("Backend:", backend)
print("Workspace:", paths.root)
print("COLMAP status:", colmap_stage_status(paths))

## 1. 特徴点抽出

特徴点抽出手法の1つであるSIFTは、別の視点でも見つけやすい点を特徴点(`keypoint`)として検出し、周囲の見え方を特徴量(`descriptor`)で表します。

画像上で特徴点が被写体と画像全体に分布しているか確認します。テクスチャが少ない領域やぼけ領域では特徴点が少なくなります。

In [ ]:
if colmap_stage_status(paths)["features"]:
    print("Feature extraction already complete")
else:
    extract_features(
        paths,
        backend=backend,
        camera_model=cfg.colmap.camera_model,
        single_camera=cfg.colmap.single_camera,
        use_gpu=cfg.colmap.use_gpu,
    )

In [ ]:
colmap_database_summary(paths.database)

In [ ]:
feature_name, keypoints = read_feature_keypoints(paths.database)
feature_keypoints_figure(paths.image_dir / feature_name, keypoints);

## 2. 特徴点マッチング

特徴量が似た特徴点を対応付け、幾何的に整合しない対応をRANSACで除きます。残った対応が`verified match`です。

マッチングした特徴点を結び、画像全体に分布しているか確認します。`verified_pairs`と`verified_matches`が少ない場合は、画像間の重なりが不足しています。

In [ ]:
if colmap_stage_status(paths)["matching"]:
    print("Feature matching already complete")
else:
    match_features(paths, backend=backend, matcher=cfg.colmap.matcher, use_gpu=cfg.colmap.use_gpu)

In [ ]:
colmap_database_summary(paths.database)

In [ ]:
name_a, name_b, keypoints_a, keypoints_b, matches = read_strongest_verified_pair(paths.database)
verified_matches_figure(
    paths.image_dir / name_a, paths.image_dir / name_b, keypoints_a, keypoints_b, matches
);

## 3. カメラパラメータと3D点群の再構成

対応の強い画像ペアから再構成を始め、画像（カメラパラメータ）と3D点を順に追加します。最後にバンドル調整でカメラパラメータと3D点群を調整します。

- `registered_images`: 一貫した3D座標系へ配置できた画像数
- `sparse_points`: 複数視点から三角測量できた（画像ペア間の対応が分かった）特徴点数
- `mean_reprojection_error_px`: 推定3D点を推定カメラパラメータで画像へ投影した位置と観測位置の平均誤差（再投影誤差）

登録画像が多く、再投影誤差が小さいことを確認します。

In [ ]:
if colmap_stage_status(paths)["mapping"]:
    print("Mapping already complete")
    model_dir = select_sparse_model(paths)
else:
    model_dir = map_sparse(paths, backend=backend)

In [ ]:
sparse_model_summary(model_dir)

In [ ]:
if not colmap_stage_status(paths)["sparse_text"]:
    export_sparse_text(paths, backend=backend, model_dir=model_dir)
mapped_scene = load_colmap_text(paths.sparse_text_dir)
sparse_scene_figure(mapped_scene).show()

## 4. 画像の歪み補正

スマートフォン画像にはradial distortionがあります。COLMAPの推定値を使い、画像とカメラパラメータを歪みのないPINHOLE modelへ変換します。

左右比較で、主に画像周辺の画角や直線が変化することを確認します。

In [ ]:
if colmap_stage_status(paths)["undistortion"]:
    print("Image undistortion already complete")
else:
    undistort_images(paths, backend=backend, model_dir=model_dir)
preview_name = mapped_scene.images[0].name
original_path = paths.image_dir / preview_name
undistorted_path = paths.processed_dir / "images" / preview_name
figure, axes = plt.subplots(1, 2, figsize=(16, 6))
for axis, path, title in zip(axes, [original_path, undistorted_path], ["Original", "Undistorted"]):
    axis.imshow(plt.imread(path))
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()

## 5. 学習用modelの出力

COLMAP modelを読みやすいテキスト形式へ変換します。推定結果自体は変わりません。

- `cameras.txt`: 画像サイズとカメラ内部パラメータ
- `images.txt`: カメラ外部パラメータと3D点群に対応する特徴点座標
- `points3D.txt`: 各3D点の位置と色、対応する特徴点

出力したmodelを再読込し、カメラパラメータと3D点群を正しく読み込めることを確認します。

In [ ]:
if not colmap_stage_status(paths)["processed_text"]:
    export_processed_text(paths, backend=backend)
scene = load_colmap_text(paths.text_model_dir)
scene_summary(scene)

In [ ]:
print("COLMAP status:", colmap_stage_status(paths))